In [4]:
# import add_parent_dir
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import hstack, kron, eye, csc_matrix, block_diag
# from add_parent_dir import *
# from code_library import *
# import numpy as np
# from CliffordOps import *
from cliffordopt import *
# from NHow import *
import stim
# import numpy as np
from typing import List

## repetition code
def repCode(r,closed=True):
    '''Generate classical repetition code on r bits.
    If closed, include one dependent row closing the loop.'''
    s = r if closed else r-1 

    SX = ZMatZeros((s,r))
    for i in range(s):
        SX[i,i] = 1
        SX[i,(i+1)%r] = 1
    return SX

## Symmetric Hypergraph Product Code
def SHPC(T):
    '''Make symmetric hypergraph product code from T.
    T can either be a string or np array.
    Returns SX, SZ.'''
    T = bin2ZMat(T)
    H = matMul(T.T, T,2)
    return HPC(H,H)

def HPC(A,B):
    '''Make hypergraph product code from clasical codes A, B
    A and B can either be a string or np array.
    Returns SX, SZ.'''
    A = bin2ZMat(A)
    B = bin2ZMat(B)
    ma,na = np.shape(A)
    mb,nb = np.shape(B)
    ## Generate SX
    C = np.kron(A,ZMatI(mb))
    D = np.kron(ZMatI(ma),B)
    SX = np.hstack([C,D])
    ## Generate SZ
    C = np.kron(ZMatI(na) ,B.T)
    D = np.kron(A.T,ZMatI(nb))
    SZ = np.hstack([C,D])
    return SX, SZ

## build 2D toric code from repetition code and SHPC constr
def toric2D(r):
    '''Generate distance r 2D toric code using SHCP construction.
    Returns SX, SZ.'''
    A = repCode(r,closed=False)
    return SHPC(A)


def parity_check_matrix_to_stabilizers(matrix: np.ndarray) -> List[stim.PauliString]:
    num_rows, num_cols = matrix.shape
    assert num_cols % 2 == 0
    num_qubits = num_cols // 2

    matrix = matrix.astype(np.bool_)  # indicate the data isn't bit packed
    return [
        stim.PauliString.from_numpy(
            xs=matrix[row, :num_qubits],
            zs=matrix[row, num_qubits:],
        )
        for row in range(num_rows)
    ]
def parity_check_matrix_to_encoder(matrix: np.ndarray) -> stim.Circuit:
    stabilizers = parity_check_matrix_to_stabilizers(matrix)
    tableau = stim.Tableau.from_stabilizers(
        stabilizers,
        allow_underconstrained=True,
        allow_redundant=True
    )
    return tableau.to_circuit(method='elimination')

def symp_gens_toric(L):
    HX,HZ=toric2D(L)
    zeros = np.zeros_like(HX)
    return parity_check_matrix_to_encoder(np.array(np.vstack((np.hstack((HX,zeros)),np.hstack((zeros,HZ)))),dtype=int))

def CSS2Tableau(HX,HZ):
    mX,n = HX.shape
    mZ = len(HZ)
    S0 = ZMatZeros((mX + mZ,n*2))
    S0[:mX,:n] = HX 
    S0[mX:,n:] = HZ
    T = CodeTableau(S0)
    return T

def ToricTableau(L):
    HX,HZ=toric2D(L)
    return CSS2Tableau(HX,HZ)

def indepPaulis(S):
    '''Return an independent set of Paulis from S in symplectic form'''
    S = np.flip(S,axis=0)
    N=2
    ## RREF plus transformation matrix
    H, U = getHU(S,N,tB=2)
    ## K is a list of linear combinations of rows which result in zero
    ix = np.sum(H,axis=-1) == 0
    K = U[ix,:]
    ## RREF - so combinations of lowest weight rows are to the RHS
    K, LI = getH(K,N,retPivots=True)
    ix = invRange(len(S),LI)
    ix = np.flip(ix)
    S = S[ix,:]
    return S

def CodeTableau(S0):
    '''Return n,k tableau plus phases for stabilisers in binary form S0'''
    n = len(S0.T) // 2
    S0 = indepPaulis(S0)
    ## RREF mod 2 - only consider first n columns, return pivots
    H, Li = getH(S0,2,nC=n,retPivots=True)
    ## independent X checks
    r = len(Li)
    ## reorder rows so pivots are to LHS
    ix = ZMat(Li + invRange(n,Li))
    H = ZMatPermuteCols(H,ix,tB=2)
    ## Swap cols r to n from X to Z component
    H = XZhad(H,np.arange(r,n))
    ## RREF again
    H,Li = getH(H,2,nC=n,retPivots=True)
    ## number of independent Z checks
    s = len(Li) - r
    ## number of encoded qubits
    k = n - r - s
    ## reorder columns
    ix2 = Li + invRange(n,Li)
    ix = ix[ix2]
    H = ZMatPermuteCols(H,ix2,tB=2)
    ## swap back cols r to n from X to Z component
    H = XZhad(H,np.arange(r,n))
    ## Extract key matrices
    A2 = H[:r,r+s:n]
    C = H[:r,-k:]
    E = H[r:,-k:]
    ## Form LX/LZ
    LX = ZMatHstack([ZMatZeros((k,r)),E.T,ZMatI(k),C.T, ZMatZeros((k,s+k))])
    LZ = ZMatHstack([ZMatZeros((k,n)),A2.T,ZMatZeros((k,s)),ZMatI(k)])
    ## Form destabilisers
    Rx = ZMatHstack([ZMatZeros((r,n)),ZMatI(r),ZMatZeros((r,n-r))])
    Rz = ZMatHstack([ZMatZeros((s,r)),ZMatI(s),ZMatZeros((s,n+k))])
    R = ZMatVstack([Rx,Rz])
    ## Find transformation U from S0 to H
    S1 = ZMatPermuteCols(S0,ix,tB=2)
    r0, U = HowResU(S1,H,2)
    ## Apply transformation U.T to destabilisers
    R = matMul(U.T,R,2)
    ## return qubits to original order
    ixR = ixRev(ix)
    ## Stack to form tableau, return qubits to original order
    T = ZMatVstack([R,LX,S1,LZ])
    T = ZMatPermuteCols(T,ixR,tB=2)
    return T

# print(badtoric_enc)
# MatList = [Stab2Tableau(S0)[-1]]
MatList=[ToricTableau(L) for L in range(2,8)]
# circlist=[symp_gens_toric(2).to_qasm(open_qasm_version=2),symp_gens_toric(3).to_qasm(open_qasm_version=2),symp_gens_toric(4).to_qasm(open_qasm_version=2),symp_gens_toric(5).to_qasm(open_qasm_version=2)]

params = paramObj()
params.minDepth = True
params.mode = 'sym'

params.method = 'greedy'
# params.method = 'astar'
# params.method = 'volanto'
# params.method = 'rustiq'
# params.method = 'stim'
# params.method = 'pyzx'

## Qiskit: methodName in ['greedy','ag']
# params.method = 'qiskit'
# params.methodName = "greedy"

## Pytket: methodName in ['FullPeepholeOptimise','CliffordSimp','SynthesiseTket','CliffordResynthesis']
# params.method = 'pytket'
# params.methodName = "FullPeepholeOptimise"


params.hv = 1 ## vector
params.hi = 1 ## include inverse
params.ht = 1 ## include transpose
params.hl = 1 ## log of cols 1 or sums 0
params.wMax = 0
params.qMax = 1000 # max priority queue length 
params.hr = 3 # scaling factor for heuristic

# gateCount,depth,t,c,check = synth_QC(circlist[3],params)
for A in MatList:
    n,gateCount,depth,procTime,check,circ = synth_Sp(A,params)
    print(f'n: {n}')
    print(f'Entangling Gate Count: {gateCount}')
    print(f'Circuit Depth: {depth}')
    print(f'Processing time: {procTime}')
    if check != "":
        print(f'Check: {check}')
    print(circ)


n: 8
Entangling Gate Count: 13
Circuit Depth: 7
Processing time: 0.19794816698413342
Check: True
Z:2 Y:3 X:7 QPerm:2,1,5,6,0,4,3,7 HS:0 H:1 HSH:2 SH:3 HSH:5 S:6 H:7 tXY:4,6 tXX:4,5 tXX:2,4 tXX:0,4 tZX:7,4 tXY:5,7 tZY:0,6 tXY:2,3 tZY:1,4 tZX:0,7 tZX:6,2 tZX:4,5 tZX:1,3
n: 18
Entangling Gate Count: 38
Circuit Depth: 9
Processing time: 0.3928644580009859
Check: True
X:1 X:3 Z:4 Z:5 Z:6 Z:7 Z:8 Y:11 X:12 Y:13 Y:15 X:16 Y:17 QPerm:3,4,5,6,7,2,15,14,16,1,0,13,8,9,10,11,12,17 S:0 HS:1 HSH:2 HSH:3 S:4 SH:5 S:6 HSH:7 S:8 HS:10 S:11 S:12 S:13 HSH:14 S:15 S:16 H:17 tYY:10,12 tYY:6,10 tYY:3,10 tZY:13,6 tXY:13,10 tZY:3,6 tXY:9,12 tZY:4,10 tYY:0,6 tZY:3,15 tXX:1,9 tZY:5,11 tXY:0,10 tXY:4,15 tZY:16,6 tZX:17,9 tZY:5,2 tZY:0,12 tZX:5,13 tYY:9,10 tYY:11,17 tZY:8,6 tZX:15,7 tZX:2,14 tZY:9,4 tZX:2,13 tZX:5,8 tZX:2,8 tXY:16,10 tZX:6,7 tZX:11,14 tZY:1,12 tZX:0,13 tZX:3,16 tZX:4,17 tZX:5,10 tZX:1,14 tZX:12,7
n: 32
Entangling Gate Count: 90
Circuit Depth: 18
Processing time: 2.9219056659785565
Check: True
X:0